# Task 75: Confidence-Aware Prediction Explanations

Generate a human-readable explanation from exact linear-model logit contributions and adapt language to confidence without presenting associations as causes.

In [1]:
from pathlib import Path
import html
import numpy as np
import pandas as pd

paths=[Path('heart.csv'),Path('Task_75_Confidence_Aware_Explanations/heart.csv'),Path('Heart_Disease_Tasks/Task_75_Confidence_Aware_Explanations/heart.csv')]
data=pd.read_csv(next(p for p in paths if p.is_file())).drop_duplicates().reset_index(drop=True)
features=[c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X,y=data[features].to_numpy(float),data.target.to_numpy(int)
rng=np.random.default_rng(75);train=[];test=[]
for label in np.unique(y):
    ids=np.flatnonzero(y==label);rng.shuffle(ids);cut=round(.8*len(ids));train.extend(ids[:cut]);test.extend(ids[cut:])
train,test=np.array(train),np.array(test);rng.shuffle(train);rng.shuffle(test)
Xtr,Xte=X[train],X[test];ytr,yte=y[train],y[test]
mean,scale=Xtr.mean(0),Xtr.std(0);scale[scale==0]=1;Xtr,Xte=(Xtr-mean)/scale,(Xte-mean)/scale
weights=np.zeros(Xtr.shape[1]);bias=0.
for _ in range(1400):
    probability=1/(1+np.exp(-np.clip(Xtr@weights+bias,-35,35)));error=probability-ytr
    weights-=.06*(Xtr.T@error/len(ytr)+.02*weights);bias-=.06*error.mean()
case_index=0;row=Xte[case_index];probability=float(1/(1+np.exp(-np.clip(row@weights+bias,-35,35))))
contributions=weights*row;confidence=max(probability,1-probability);predicted='positive' if probability>=.5 else 'negative'
if confidence>=.8:confidence_text='The model is relatively confident in this classification.'
elif confidence>=.65:confidence_text='The model has moderate confidence; review the contributing inputs.'
else:confidence_text='The model is uncertain. Do not rely on this result without further assessment.'
order=np.argsort(np.abs(contributions))[::-1][:3]
reasons=[]
for index in order:
    effect='increases' if contributions[index]>0 else 'decreases'
    reasons.append(f'{html.escape(features[index])} {effect} the model score relative to the training mean')
explanation=f'The model predicts the {predicted} class with probability {probability:.1%}. {confidence_text} Largest score contributions: '+ '; '.join(reasons)+'. These are model associations, not causes.'
print(explanation)
report=pd.DataFrame({'Feature':features,'Standardized value':row,'Weight':weights,'Logit contribution':contributions}).sort_values('Logit contribution',key=np.abs,ascending=False)
display(report.round(3))
print(f'Intercept baseline logit: {bias:.3f}; predicted confidence max(p, 1-p): {confidence:.3f}')

The model predicts the positive class with probability 65.4%. The model has moderate confidence; review the contributing inputs. Largest score contributions: cp increases the model score relative to the training mean; thal decreases the model score relative to the training mean; sex decreases the model score relative to the training mean. These are model associations, not causes.


,Feature,Standardized value,Weight,Logit contribution
2,cp,1.065,0.708,0.753
12,thal,1.117,-0.434,-0.485
1,sex,0.659,-0.633,-0.418
7,thalach,0.577,0.566,0.326
6,restecg,0.919,0.286,0.262
8,exang,-0.685,-0.380,0.260
10,slope,-0.671,0.274,-0.184
11,ca,0.278,-0.631,-0.175
9,oldpeak,-0.366,-0.309,0.113
3,trestbps,0.431,-0.195,-0.084


Intercept baseline logit: 0.122; predicted confidence max(p, 1-p): 0.654
